<a href="https://colab.research.google.com/github/dmousav1/CaRTS/blob/main/MLDL_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Save Session State with Dill

In [2]:
!pip install dill  # Install dill

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.4/119.4 kB 4.7 MB/s eta 0:00:00


In [3]:
    # How to save session state so everytime runtime disconnects I don't have to retrain the model
    import dill
    import os
    from google.colab import drive

    backup_dir = 'drive/My Drive/colab_sessions'
    backup_file = 'mldl_env.db'
    backup_path = os.path.join(backup_dir, backup_file)

    def init_drive():
        drive.mount('drive')
        os.makedirs(backup_dir, exist_ok=True)

    def save_session():
        init_drive()
        dill.dump_session(backup_path)

    def load_session():
        init_drive()
        dill.load_session(backup_path)

In [4]:
    import pickle
    # Save the model
    with open('model.pkl', 'wb') as file:
        pickle.dump(model, file)

    # Load the model
    with open('model.pkl', 'rb') as file:
        loaded_model = pickle.load(file)

NameError: name 'model' is not defined

In [5]:
    #
    from google.colab import files
    files.download('my_data.csv')

FileNotFoundError: Cannot find file: my_data.csv

# Docker System Setup and Installing Python Packages

In [6]:
# Switch from running on CPU to GPU
import torch
if torch.cuda.is_available():
    device_name = torch.device("cuda")
else:
    device_name = torch.device('cpu')
print("Using {}.".format(device_name))

Using cpu.


In [8]:
!pip install numpy pandas tensorflow keras  # add all packages you need

In [ ]:
!apt-get update
!apt-get install -y <package-name>

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,688 kB]
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [2,788 kB]
Hit:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [3,099 kB

# Load Training, Validation, and Testing Data

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
# Write training, validation, and testing dataset file pathways

data_path = '/content/drive/MyDrive/DaVinci_SegStrongC_Challenge'
training_path = f'{data_path}/SegSTRONGC_train/SegSTRONGC_train'
validation_path = f'{data_path}/SegSTRONGC_val/SegSTRONGC_val/val'
testing_path = f'{data_path}/SegSTRONGC_test/SegSTRONGC_test/test'

In [11]:
# Loading Training Data

import os
import tensorflow as tf

def load_uncorrupted_image_paths_by_category(base_dir):
    """
    Walks through the folder structure starting at base_dir.
    Expects a structure like:
        <base_dir>/<batch>/<minibatch>/
            ground_truth/left/
            ground_truth/right/
            regular/left/
            regular/right/

    Returns a dictionary with keys 'regular' and 'ground_truth',
    each containing sub-keys 'left' and 'right' holding the full file paths.
    """
    data = {
        'ground_truth': {'left': [], 'right': []},
        'regular': {'left': [], 'right': []}
    }

    # Walk through the directory
    for root, dirs, files in os.walk(base_dir):
        current_folder = os.path.basename(root).lower()  # e.g., "left" or "right"
        parent_folder = os.path.basename(os.path.dirname(root)).lower()  # should be "ground_truth" or "regular"

        # Check if we are in one of the target subdirectories:
        if parent_folder in ['ground_truth', 'regular'] and current_folder in ['left', 'right']:
            for file in files:
                # Check case-insensitively for image extensions (png or jpg)
                if file.lower().endswith('.png') or file.lower().endswith('.jpg'):
                    full_path = os.path.join(root, file)
                    data[parent_folder][current_folder].append(full_path)

                    # Optional: print each file found for debugging
                    # print(f"Found file: {full_path}")

    return data

# Set the base path to your training data folder.
# Adjust the path as necessary (note the duplicate 'SegSTRONGC_train' if that's indeed your structure)
train_data = load_uncorrupted_image_paths_by_category(training_path)

# I'll uncomment this when I do the validation and training data loading
'''
validation_data = load_image_paths_by_category(validation_path)
testing_data = load_image_paths_by_category(testing_path)
'''

# Print out a summary of the loaded data.
# Training Data
print("Training Data Loaded:")
print(" - Ground Truth Left:", len(train_data['ground_truth']['left']))
print(" - Ground Truth Right:", len(train_data['ground_truth']['right']))
print(" - Regular Left:", len(train_data['regular']['left']))
print(" - Regular Right:", len(train_data['regular']['right']))

# I'll uncomment this when I do the validation and training data loading
'''
# Validation Data
print("Validation Data Loaded:")
print(" - Ground Truth Left:", len(validation_data['ground_truth']['left']))
print(" - Ground Truth Right:", len(validation_data['ground_truth']['right']))
print(" - Regular Left:", len(validation_data['regular']['left']))
print(" - Regular Right:", len(validation_data['regular']['right']))

# Testing Data
print("Testing Data Loaded:")
print(" - Ground Truth Left:", len(testing_data['ground_truth']['left']))
print(" - Ground Truth Right:", len(testing_data['ground_truth']['right']))
print(" - Regular Left:", len(testing_data['regular']['left']))
print(" - Regular Right:", len(testing_data['regular']['right']))
'''



Training Data Loaded:
 - Ground Truth Left: 3300
 - Ground Truth Right: 3300
 - Regular Left: 3300
 - Regular Right: 3300


'\n# Validation Data\nprint("Validation Data Loaded:")\nprint(" - Ground Truth Left:", len(validation_data[\'ground_truth\'][\'left\']))\nprint(" - Ground Truth Right:", len(validation_data[\'ground_truth\'][\'right\']))\nprint(" - Regular Left:", len(validation_data[\'regular\'][\'left\']))\nprint(" - Regular Right:", len(validation_data[\'regular\'][\'right\']))\n\n# Testing Data\nprint("Testing Data Loaded:")\nprint(" - Ground Truth Left:", len(testing_data[\'ground_truth\'][\'left\']))\nprint(" - Ground Truth Right:", len(testing_data[\'ground_truth\'][\'right\']))\nprint(" - Regular Left:", len(testing_data[\'regular\'][\'left\']))\nprint(" - Regular Right:", len(testing_data[\'regular\'][\'right\']))\n'

In [15]:
# Loading Validation Data
def load_corrupted_image_paths_by_category(base_dir):
    """
    Walks through the folder structure starting at base_dir.
    Expects a structure like:
        <base_dir>/<batch>/<minibatch>/
            ground_truth/left/
            ground_truth/right/
            regular/left/
            regular/right/
            smoke/left/
            smoke/right/
            low_brightness/left/
            low_brightness/right/
            blood/left/
            blood/right/
            bg_change/left/
            bg_change/right/

    Returns a dictionary with keys 'regular', 'ground_truth', 'smoke',
    'low_brightness', 'blood', 'bg_change',
    each containing sub-keys 'left' and 'right' holding the full file paths.
    """
    data = {
        'ground_truth': {'left': [], 'right': []},
        'regular': {'left': [], 'right': []},
        'smoke': {'left': [], 'right': []},
        'low_brightness': {'left': [], 'right': []},
        'blood': {'left': [], 'right': []},
        'bg_change': {'left': [], 'right': []}
    }

    # Walk through the directory
    for root, dirs, files in os.walk(base_dir):
        current_folder = os.path.basename(root).lower()  # e.g., "left" or "right"
        parent_folder = os.path.basename(os.path.dirname(root)).lower()  # should be keys listed above

        # Check if we are in one of the target subdirectories:
        if parent_folder in ['ground_truth', 'regular', 'smoke', 'low_brightness', 'blood', 'bg_change'] and current_folder in ['left', 'right']:
            for file in files:
                # Check case-insensitively for image extensions (png or jpg)
                if file.lower().endswith('.png'):
                    full_path = os.path.join(root, file)
                    data[parent_folder][current_folder].append(full_path)

                    # Optional: print each file found for debugging
                    # print(f"Found file: {full_path}")

    return data

# Set the base path to your validation and testing data folder.
validation_data = load_corrupted_image_paths_by_category(validation_path)
testing_data = load_corrupted_image_paths_by_category(testing_path)

# Print out a summary of the loaded data.
# Validation Data
print("Validation Data Loaded:")
print(" - Ground Truth Left:", len(validation_data['ground_truth']['left']))
print(" - Ground Truth Right:", len(validation_data['ground_truth']['right']))
print(" - Regular Left:", len(validation_data['regular']['left']))
print(" - Regular Right:", len(validation_data['regular']['right']))
print(" - Smoke Left:", len(validation_data['smoke']['left']))
print(" - Smoke Right:", len(validation_data['smoke']['right']))
print(" - Low Brightness Left:", len(validation_data['low_brightness']['left']))
print(" - Low Brightness Right:", len(validation_data['low_brightness']['right']))
print(" - Blood Left:", len(validation_data['blood']['left']))
print(" - Blood Right:", len(validation_data['blood']['right']))
print(" - Background Change Left:", len(validation_data['bg_change']['left']))
print(" - Background Change Right:", len(validation_data['bg_change']['right']))

# Testing Data
print("Testing Data Loaded:")
print(" - Ground Truth Left:", len(testing_data['ground_truth']['left']))
print(" - Ground Truth Right:", len(testing_data['ground_truth']['right']))
print(" - Regular Left:", len(testing_data['regular']['left']))
print(" - Regular Right:", len(testing_data['regular']['right']))
print(" - Smoke Left:", len(testing_data['smoke']['left']))
print(" - Smoke Right:", len(testing_data['smoke']['right']))
print(" - Low Brightness Left:", len(testing_data['low_brightness']['left']))
print(" - Low Brightness Right:", len(testing_data['low_brightness']['right']))
print(" - Blood Left:", len(testing_data['blood']['left']))
print(" - Blood Right:", len(testing_data['blood']['right']))
print(" - Background Change Left:", len(testing_data['bg_change']['left']))
print(" - Background Change Right:", len(testing_data['bg_change']['right']))

Validation Data Loaded:
 - Ground Truth Left: 900
 - Ground Truth Right: 900
 - Regular Left: 900
 - Regular Right: 900
 - Smoke Left: 900
 - Smoke Right: 900
 - Low Brightness Left: 900
 - Low Brightness Right: 900
 - Blood Left: 900
 - Blood Right: 900
 - Background Change Left: 900
 - Background Change Right: 900
Testing Data Loaded:
 - Ground Truth Left: 900
 - Ground Truth Right: 900
 - Regular Left: 900
 - Regular Right: 900
 - Smoke Left: 900
 - Smoke Right: 900
 - Low Brightness Left: 900
 - Low Brightness Right: 900
 - Blood Left: 900
 - Blood Right: 900
 - Background Change Left: 900
 - Background Change Right: 900


# Model 1: U-Net for Segmentation

In [ ]:
!pip install torch torchvision # Needed for setting up our 2 model pipeline
!nvidia-smi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 127.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 106.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 106.7 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvji

/bin/bash: line 1: nvidia-smi: command not found


In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# A helper module: two consecutive conv layers with BatchNorm and ReLU
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, features=[64, 128, 256, 512, 1024]):
        """
        in_channels: number of input image channels (e.g., 3 for RGB)
        out_channels: segmentation mask channels (1 for binary segmentation)
        features: list defining channel size at each encoder block
        """
        super(UNet, self).__init__()

        self.downs = nn.ModuleList()
        self.pools = nn.ModuleList()
        # Encoder: 5 blocks
        prev_channels = in_channels
        for feature in features:
            self.downs.append(DoubleConv(prev_channels, feature))
            self.pools.append(nn.MaxPool2d(kernel_size=2, stride=2))
            prev_channels = feature

        # Bottleneck block: double the last encoder feature channels
        self.bottleneck = DoubleConv(features[-1], features[-1]*2)

        # Decoder: for each encoder block (in reverse)
        self.ups = nn.ModuleList()
        reversed_features = features[::-1]
        curr_channels = features[-1]*2  # input from bottleneck

        for feature in reversed_features:
            # Transposed convolution to upsample
            self.ups.append(
                nn.ConvTranspose2d(curr_channels, feature, kernel_size=2, stride=2)
            )
            # After concatenation (skip connection + upsampled feature), we have "feature*2" channels
            self.ups.append(DoubleConv(feature*2, feature))
            curr_channels = feature

        # Final 1x1 conv to obtain desired segmentation output
        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def forward(self, x):
        skip_connections = []

        # Encoder
        for idx in range(len(self.downs)):
            x = self.downs[idx](x)
            skip_connections.append(x)
            x = self.pools[idx](x)

        # Bottleneck
        x = self.bottleneck(x)
        # Reverse the order of skip connections for the decoder
        skip_connections = skip_connections[::-1]

        # Decoder
        for idx in range(0, len(self.ups), 2):
            x = self.ups[idx](x)  # upsample: ConvTranspose2d
            skip_connection = skip_connections[idx // 2]
            # Sometimes, due to rounding, dimensions might not match exactly—resize if necessary:
            if x.shape[2:] != skip_connection.shape[2:]:
                x = F.interpolate(x, size=skip_connection.shape[2:], mode='bilinear', align_corners=True)
            # Concatenate along channel axis
            x = torch.cat((skip_connection, x), dim=1)
            x = self.ups[idx+1](x)  # double conv on concatenated feature maps

        return self.final_conv(x)

# Example usage: create a UNet instance and run a dummy input through it.
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_unet = UNet(in_channels=3, out_channels=1).to(device)
    dummy_input = torch.randn(1, 3, 256, 256).to(device)
    seg_output = model_unet(dummy_input)
    print("U-Net output shape:", seg_output.shape)


U-Net output shape: torch.Size([1, 1, 256, 256])


# Model 2: ResNet-Based Binary Classifier

# Turn training data dictionary into a TensorFlow data pipeline wtih I/O pairs as 'regular'/'ground_truth'

In [21]:
# Step 1: Create custom PyTorch Dataset for training. I/O pairs of 'regular' and 'ground_truth'
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms

class SegmentationDataset(Dataset):
    def __init__(self, regular_paths, gt_paths, transform=None, target_transform=None):
        """
        regular_paths: list of file paths for the input (regular) images.
        gt_paths: list of file paths for the corresponding ground truth masks.
        transform: a torchvision transform to apply to the input image.
        target_transform: a torchvision transform to apply to the ground truth mask.
        """
        # Sort the paths to align the image order (assuming naming conventions enforce correct order)
        self.regular_paths = sorted(regular_paths)
        self.gt_paths = sorted(gt_paths)
        self.transform = transform
        self.target_transform = target_transform

        # Ensure the number of input images equals the number of ground truth masks.
        assert len(self.regular_paths) == len(self.gt_paths), "Mismatch between regular and ground truth images"

    def __len__(self):
        return len(self.regular_paths)

    def __getitem__(self, idx):
        # Load the regular image and the ground truth mask using PIL
        reg_img = Image.open(self.regular_paths[idx]).convert('RGB')
        gt_img = Image.open(self.gt_paths[idx]).convert('L')  # ground truth is single-channel

        # Apply any specified transforms
        if self.transform:
            reg_img = self.transform(reg_img)
        if self.target_transform:
            gt_img = self.target_transform(gt_img)

        return reg_img, gt_img


In [26]:
# Step 2: Define the Transformations and Build a DataLoader

# Define the transforms to be applied to the images.
# For the input image:
img_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),  # Converts from HxWxC in [0,255] to CxHxW in [0,1]
    # You can add normalization here if desired (e.g., transforms.Normalize(mean, std))
])
# For the ground truth mask (we want a single channel tensor):
mask_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),  # This will give a tensor in [0,1]
])

# Use the 'left' camera view for training:
regular_left_paths = train_data['regular']['left']
gt_left_paths = train_data['ground_truth']['left']

# Create the dataset:
train_dataset = SegmentationDataset(regular_left_paths, gt_left_paths,
                                     transform=img_transform,
                                     target_transform=mask_transform)

# Create the DataLoader:
batch_size = 8  # Adjust based on your GPU memory and batch requirements.
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)

# Debug: iterate one batch to check shapes.
for images, masks in train_loader:
    print("Regular images batch shape:", images.shape)
    print("Ground truth masks batch shape:", masks.shape)
    break


Regular images batch shape: torch.Size([8, 3, 256, 256])
Ground truth masks batch shape: torch.Size([8, 1, 256, 256])


In [25]:
# Helper function for Dice loss

def dice_loss(pred, target, smooth=1):
    """
    Calculate the Dice loss.

    Args:
      pred: the prediction tensor (logits) from the model.
      target: the ground truth segmentation mask tensor.
      smooth: a smoothing factor to avoid division by zero.

    Returns:
      A scalar Dice loss.
    """
    # Apply sigmoid to get probabilities from logits.
    pred = torch.sigmoid(pred)

    # Flatten the tensors to simplify the computation.
    pred = pred.contiguous().view(-1)
    target = target.contiguous().view(-1)

    # Calculate intersection and Dice coefficient.
    intersection = (pred * target).sum()
    dice_coeff = (2. * intersection + smooth) / (pred.sum() + target.sum() + smooth)

    # Dice loss is defined as 1 - Dice coefficient.
    return 1 - dice_coeff

In [24]:
# Step 3: Integrate with U-Net Model Training Loop

import torch.optim as optim

# U-Net model implemented above
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_unet = UNet(in_channels=3, out_channels=1).to(device)

# Define loss function and optimizer. We switch loss criterion from BCE/Sigmoid to Dice
# for better segmentation
# criterion = nn.BCEWithLogitsLoss()  # Applies sigmoid internally
optimizer = optim.Adam(model_unet.parameters(), lr=1e-4)

num_epochs = 10  # Set your number of training epochs

for epoch in range(num_epochs):
    model_unet.train()
    running_loss = 0.0
    for images, masks in train_loader:
        # Move data to GPU if available.
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()  # Zero the gradients
        outputs = model_unet(images)

        # Compute the loss
        loss = dice_loss(outputs, masks)

        loss.backward()  # Back-propagate the gradients
        optimizer.step()  # Update model parameters

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_dataset)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}")


KeyboardInterrupt: 